In [ ]:
# ЛАБОРАТОРНАЯ РАБОТА 6. СЕМАНТИЧЕСКАЯ СЕГМЕНТАЦИЯ

# Шаг 0: Подготовка окружения (установка библиотек и распаковка данных)

!pip install -q albumentations segmentation-models-pytorch torchsummary

# Распакуйте архив с данными (замените имя файла, если оно отличается)
!unzip -q dataset.zip

# Шаг 1: Импорт всех необходимых библиотек

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
from tqdm.notebook import tqdm

# Scikit-learn
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torch.nn.functional as F

# Albumentations
import albumentations as A

# Работа с изображениями
from PIL import Image
import cv2

# Segmentation Models PyTorch
import segmentation_models_pytorch as smp
from torchsummary import summary

# Шаг 2: Настройка устройства и путей к данным

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

IMAGE_PATH = 'dataset/semantic_drone_dataset/original_images/'
MASK_PATH = 'dataset/semantic_drone_dataset/label_images_semantic/'

# Шаг 3: Создание DataFrame и разделение на выборки

n_classes = 23

def create_df():
    """Создает DataFrame со списком ID всех изображений."""
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            name.append(filename.split('.')[0])
    return pd.DataFrame({'id': name}, index=np.arange(0, len(name)))

df = create_df()
print('Total Images: ', len(df))

# Разделение на train+val и test (10% данных уходит в тест)
X_trainval, X_test = train_test_split(df['id'].values, test_size=0.1, random_state=19)

# Разделение train+val на train и val (15% от trainval уходит в val)
X_train, X_val = train_test_split(X_trainval, test_size=0.15, random_state=19)

print(f'Train Size: {len(X_train)}')
print(f'Val Size: {len(X_val)}')
print(f'Test Size: {len(X_test)}')

# Шаг 4: Визуализация примера данных (проверка загрузки)

# Возьмем первый элемент из датафрейма
img = Image.open(IMAGE_PATH + df['id'][0] + '.jpg')
mask = Image.open(MASK_PATH + df['id'][0] + '.png')
print('Image Size', np.asarray(img).shape)
print('Mask Size', np.asarray(mask).shape)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title('Original Image')
plt.subplot(1, 2, 2)
plt.imshow(img)
plt.imshow(mask, alpha=0.6)  # Накладываем маску поверх изображения
plt.title('Image with Mask Applied')
plt.show()

# Шаг 5: Создание класса Dataset для обучения

class DroneDataset(Dataset):
    """Датасет для загрузки изображений и масок."""
    
    def __init__(self, img_path, mask_path, X, mean, std, transform=None, patch=False):
        self.img_path = img_path
        self.mask_path = mask_path
        self.X = X
        self.transform = transform
        self.patches = patch
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Загрузка изображения через OpenCV
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Загрузка маски
        mask = cv2.imread(self.mask_path + self.X[idx] + '.png', cv2.IMREAD_GRAYSCALE)

        # Применение аугментаций
        if self.transform is not None:
            aug = self.transform(image=img, mask=mask)
            img = Image.fromarray(aug['image'])
            mask = aug['mask']
        else:
            img = Image.fromarray(img)

        # Нормализация и преобразование в тензор
        t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        img = t(img)
        mask = torch.from_numpy(mask).long()

        return img, mask

# Шаг 6: Аугментации и создание DataLoader'ов

# Параметры нормализации (ImageNet)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Аугментации для тренировочных данных
t_train = A.Compose([
    A.Resize(704, 1056, interpolation=cv2.INTER_NEAREST),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GridDistortion(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=(0, 0.5), contrast_limit=(0, 0.5), p=0.5),
    A.GaussNoise(p=0.2)
])

# Аугментации для валидационных данных (минимальные)
t_val = A.Compose([
    A.Resize(704, 1056, interpolation=cv2.INTER_NEAREST),
    A.HorizontalFlip(p=0.5),
    A.GridDistortion(p=0.2)
])

# Создание датасетов
train_set = DroneDataset(IMAGE_PATH, MASK_PATH, X_train, mean, std, t_train, patch=False)
val_set = DroneDataset(IMAGE_PATH, MASK_PATH, X_val, mean, std, t_val, patch=False)

# DataLoader'ы
batch_size = 3
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

# Шаг 7: Загрузка модели U-Net с бэкбоном MobileNetV2

model = smp.Unet(
    'mobilenet_v2',
    encoder_weights='imagenet',
    classes=23,
    activation=None,
    encoder_depth=5,
    decoder_channels=[256, 128, 64, 32, 16]
)

print("Модель загружена.")

# Шаг 8: Определение функций для вычисления метрик

def pixel_accuracy(output, mask):
    """Вычисляет попиксельную точность."""
    with torch.no_grad():
        output = torch.argmax(F.softmax(output, dim=1), dim=1)
        correct = torch.eq(output, mask).int()
        accuracy = float(correct.sum()) / float(correct.numel())
    return accuracy


def mIoU(pred_mask, mask, smooth=1e-10, n_classes=23):
    """Вычисляет средний Intersection over Union (mIoU)."""
    with torch.no_grad():
        pred_mask = F.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1)
        pred_mask = pred_mask.contiguous().view(-1)
        mask = mask.contiguous().view(-1)

        iou_per_class = []
        for clas in range(0, n_classes):
            true_class = pred_mask == clas
            true_label = mask == clas

            if true_label.long().sum().item() == 0:
                iou_per_class.append(np.nan)
            else:
                intersect = torch.logical_and(true_class, true_label).sum().float().item()
                union = torch.logical_or(true_class, true_label).sum().float().item()
                iou = (intersect + smooth) / (union + smooth)
                iou_per_class.append(iou)
        return np.nanmean(iou_per_class)

# Шаг 9: Функция обучения модели

def get_lr(optimizer):
    """Возвращает текущую скорость обучения."""
    for param_group in optimizer.param_groups:
        return param_group['lr']


def fit(epochs, model, train_loader, val_loader, criterion, optimizer, scheduler, patch=False):
    """Основная функция обучения."""
    torch.cuda.empty_cache()
    train_losses, test_losses = [], []
    val_iou, val_acc = [], []
    train_iou, train_acc = [], []
    lrs = []
    min_loss = np.inf
    not_improve = 0

    model.to(device)
    fit_time = time.time()
    
    for e in range(epochs):
        since = time.time()
        running_loss = 0
        iou_score = 0
        accuracy = 0

        # Фаза обучения
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            image_tiles, mask_tiles = data

            if patch:
                bs, n_tiles, c, h, w = image_tiles.size()
                image_tiles = image_tiles.view(-1, c, h, w)
                mask_tiles = mask_tiles.view(-1, h, w)

            image = image_tiles.to(device)
            mask = mask_tiles.to(device)

            # Прямой проход
            output = model(image)
            loss = criterion(output, mask)

            # Метрики
            iou_score += mIoU(output, mask)
            accuracy += pixel_accuracy(output, mask)

            # Обратное распространение
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Обновление скорости обучения
            lrs.append(get_lr(optimizer))
            scheduler.step()

            running_loss += loss.item()

        # Фаза валидации
        model.eval()
        test_loss = 0
        test_accuracy = 0
        val_iou_score = 0

        with torch.no_grad():
            for i, data in enumerate(tqdm(val_loader)):
                image_tiles, mask_tiles = data
                if patch:
                    bs, n_tiles, c, h, w = image_tiles.size()
                    image_tiles = image_tiles.view(-1, c, h, w)
                    mask_tiles = mask_tiles.view(-1, h, w)

                image = image_tiles.to(device)
                mask = mask_tiles.to(device)
                output = model(image)

                val_iou_score += mIoU(output, mask)
                test_accuracy += pixel_accuracy(output, mask)
                loss = criterion(output, mask)
                test_loss += loss.item()

        # Сохранение статистики
        train_losses.append(running_loss / len(train_loader))
        test_losses.append(test_loss / len(val_loader))
        val_iou.append(val_iou_score / len(val_loader))
        train_iou.append(iou_score / len(train_loader))
        train_acc.append(accuracy / len(train_loader))
        val_acc.append(test_accuracy / len(val_loader))

        # Ранняя остановка и сохранение модели
        if min_loss > (test_loss / len(val_loader)):
            print(f'Loss Decreasing.. {min_loss:.3f} >> {(test_loss/len(val_loader)):.3f}')
            min_loss = (test_loss / len(val_loader))
            not_improve = 0
            if len(test_losses) % 5 == 0:
                print('Saving model...')
                torch.save(model, f'Unet-Mobilenet_v2_mIoU-{val_iou[-1]:.3f}.pt')
        else:
            not_improve += 1
            print(f'Loss Not Decrease for {not_improve} time')
            if not_improve == 7:
                print('Loss not decrease for 7 times, Stop Training')
                break

        # Вывод информации об эпохе
        print(f"Epoch:{e+1}/{epochs}.. "
              f"Train Loss: {train_losses[-1]:.3f}.. "
              f"Val Loss: {test_losses[-1]:.3f}.. "
              f"Train mIoU:{train_iou[-1]:.3f}.. "
              f"Val mIoU: {val_iou[-1]:.3f}.. "
              f"Train Acc:{train_acc[-1]:.3f}.. "
              f"Val Acc:{val_acc[-1]:.3f}.. "
              f"Time: {(time.time()-since)/60:.2f}m")

    history = {
        'train_loss': train_losses,
        'val_loss': test_losses,
        'train_miou': train_iou,
        'val_miou': val_iou,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'lrs': lrs
    }
    print(f'Total time: {(time.time()-fit_time)/60:.2f} m')
    return history

# Шаг 10: Настройка оптимизатора и запуск обучения

max_lr = 1e-3
epoch = 15
weight_decay = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr, epochs=epoch, steps_per_epoch=len(train_loader)
)

# ЗАПУСК ОБУЧЕНИЯ
history = fit(epoch, model, train_loader, val_loader, criterion, optimizer, sched)

# Шаг 11: Визуализация процесса обучения

def plot_loss(history):
    plt.plot(history['val_loss'], label='val', marker='o')
    plt.plot(history['train_loss'], label='train', marker='o')
    plt.title('Loss per epoch')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend()
    plt.grid()
    plt.show()

def plot_score(history):
    plt.plot(history['train_miou'], label='train_mIoU', marker='*')
    plt.plot(history['val_miou'], label='val_mIoU', marker='*')
    plt.title('Score per epoch')
    plt.ylabel('mean IoU')
    plt.xlabel('epoch')
    plt.legend()
    plt.grid()
    plt.show()

def plot_acc(history):
    plt.plot(history['train_acc'], label='train_accuracy', marker='*')
    plt.plot(history['val_acc'], label='val_accuracy', marker='*')
    plt.title('Accuracy per epoch')
    plt.ylabel('Accuracy')
    plt.xlabel('epoch')
    plt.legend()
    plt.grid()
    plt.show()

plot_loss(history)
plot_score(history)
plot_acc(history)

# Шаг 12: Оценка модели на тестовой выборке

class DroneTestDataset(Dataset):
    """Датасет для тестовой выборки."""
    
    def __init__(self, img_path, mask_path, X, transform=None):
        self.img_path = img_path
        self.mask_path = mask_path
        self.X = X
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_path + self.X[idx] + '.png', cv2.IMREAD_GRAYSCALE)

        if self.transform is not None:
            aug = self.transform(image=img, mask=mask)
            img = Image.fromarray(aug['image'])
            mask = aug['mask']
        else:
            img = Image.fromarray(img)

        mask = torch.from_numpy(mask).long()
        return img, mask


def predict_image_mask_miou(model, image, mask, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Предсказывает маску и вычисляет mIoU для одного изображения."""
    model.eval()
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    image = t(image)
    model.to(device)
    image = image.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        image = image.unsqueeze(0)
        mask = mask.unsqueeze(0)
        output = model(image)
        score = mIoU(output, mask)
        masked = torch.argmax(output, dim=1).cpu().squeeze(0)
    return masked, score


def predict_image_mask_pixel(model, image, mask, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Предсказывает маску и вычисляет точность для одного изображения."""
    model.eval()
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    image = t(image)
    model.to(device)
    image = image.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        image = image.unsqueeze(0)
        mask = mask.unsqueeze(0)
        output = model(image)
        acc = pixel_accuracy(output, mask)
        masked = torch.argmax(output, dim=1).cpu().squeeze(0)
    return masked, acc


def miou_score(model, test_set):
    """Вычисляет средний mIoU по всей тестовой выборке."""
    score_iou = []
    for i in tqdm(range(len(test_set))):
        img, mask = test_set[i]
        _, score = predict_image_mask_miou(model, img, mask)
        score_iou.append(score)
    return score_iou


def pixel_acc(model, test_set):
    """Вычисляет среднюю точность по всей тестовой выборке."""
    accuracy = []
    for i in tqdm(range(len(test_set))):
        img, mask = test_set[i]
        _, acc = predict_image_mask_pixel(model, img, mask)
        accuracy.append(acc)
    return accuracy


# Тестовые аугментации (только ресайз)
t_test = A.Resize(768, 1152, interpolation=cv2.INTER_NEAREST)
test_set = DroneTestDataset(IMAGE_PATH, MASK_PATH, X_test, transform=t_test)

# Визуализация одного предсказания
image, mask = test_set[3]
pred_mask, score = predict_image_mask_miou(model, image, mask)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 10))
ax1.imshow(image)
ax1.set_title('Picture')
ax2.imshow(mask)
ax2.set_title('Ground truth')
ax3.imshow(pred_mask)
ax3.set_title(f'UNet-MobileNet | mIoU {score:.3f}')
plt.show()

# Расчет средних метрик по тестовой выборке
mob_miou = miou_score(model, test_set)
mob_acc = pixel_acc(model, test_set)

print(f'Test Set mIoU: {np.mean(mob_miou):.4f}')
print(f'Test Set Pixel Accuracy: {np.mean(mob_acc):.4f}')

print("\nОБУЧЕНИЕ И ОЦЕНКА ЗАВЕРШЕНЫ!")